## ZAD 1


In [1]:
import time
import functools
class TimerDecorator:

    def __init__(self,func):
        self.func=func  #moja funkcja
        self.czasy_wykonania=[]
        functools.update_wrapper(self, func) #WARIANT D
    
    
    def __call__(self,*args,**kwargs):
        start = time.perf_counter()
        wynik=self.func(*args,**kwargs)  #dekorator
        koniec = time.perf_counter()
        self.czasy_wykonania.append(koniec - start)
        return wynik

    def pokaz_statystyki(self):
        if len(self.czasy_wykonania) >0: #atrybuty klasy zawsze wywołujemy z self
            print(f'Statystyki dla funkcji {self.func.__name__}:')
            print(f'Liczba wywołań: {len(self.czasy_wykonania)}')
            print(f'Najkrótszy czas wykonania: {min(self.czasy_wykonania)}')
            print(f'Najdłuższy czas wykonania: {max(self.czasy_wykonania)}')
            print(f'Średni czas wykonania: {sum(self.czasy_wykonania) / len(self.czasy_wykonania)}')
        else:
            print(f'Funkcja nie została wykonana ani razu, brak pomiarów.')

    #wariant C
    def pobierz_statystyki(self):
        if len(self.czasy_wykonania) > 0: #zwracamy słownik a nie printy
            return {
                'nazwa': self.func.__name__,
                'liczba wywołań': len(self.czasy_wykonania),
                'najkrótszy czas': min(self.czasy_wykonania),
                'najdłuższy czas': max(self.czasy_wykonania),
                'średni czas': sum(self.czasy_wykonania) / len(self.czasy_wykonania)
            }
        else:
            return None
    

In [2]:
#testowanie
@TimerDecorator
def czas_test(sekundy):
    print(f'Funkcja będzie się wykonywać przez {sekundy} sekund...')
    time.sleep(sekundy)

@TimerDecorator
def dodawanie(a,b):
    '''Dodawanie lololololol'''
    return a+b

czas_test(2)
czas_test(1) 

print('-'*30)

print(dodawanie(5,10))
print(dodawanie(20,30))

print('-'*30)

czas_test.pokaz_statystyki() 

print('-'*30)

dodawanie.pokaz_statystyki()

print('-'*30)

print(dodawanie.__name__)
print(dodawanie.__doc__)
#dodanie @timerdecorator sprawia ze mozemy korzystac z metod klasy dla róznych funkcji

Funkcja będzie się wykonywać przez 2 sekund...
Funkcja będzie się wykonywać przez 1 sekund...
------------------------------
15
50
------------------------------
Statystyki dla funkcji czas_test:
Liczba wywołań: 2
Najkrótszy czas wykonania: 1.0008636000020488
Najdłuższy czas wykonania: 2.0011728999998013
Średni czas wykonania: 1.501018250000925
------------------------------
Statystyki dla funkcji dodawanie:
Liczba wywołań: 2
Najkrótszy czas wykonania: 1.8999999156221747e-06
Najdłuższy czas wykonania: 2.2999993234407157e-06
Średni czas wykonania: 2.099999619531445e-06
------------------------------
dodawanie
Dodawanie lololololol


# WARIANT D
dopóki nie użyjemy functools.update_wrapper(self, func) to wyskoczy błąd poniżej czyli jeśli dajemy dekorator to nadpisuje on jej dane własnymi przez co tracimy jej tożsamość

Więc oryginalną funkcję (func), kopiuje jej prawdziwe imię (__name__) i oryginalny opis (__doc__), a następnie nakleja te informacje na zewnątrz dekoratora (self) żeby nawet bo @ miał swoją tożsamość zapisaną

In [3]:
'''
print(dodawanie.__name__)
AttributeError                            Traceback (most recent call last)
Cell In[10], line 1
----> 1 print(dodawanie.__name__)

AttributeError: 'TimerDecorator' object has no attribute '__name__'
'''

"\nprint(dodawanie.__name__)\nAttributeError                            Traceback (most recent call last)\nCell In[10], line 1\n----> 1 print(dodawanie.__name__)\n\nAttributeError: 'TimerDecorator' object has no attribute '__name__'\n"

## ZAD 2    

In [7]:
class Temperature:
    def __init__(self, celsius): #przypisanie wartości do atrybutu klasy
        self.celsius = celsius

    @property
    def celsius(self): #getter służy do odczytywania wartości atrybutu
        return self._celsius

    @celsius.setter  # setter ustawia wartość
    def celsius(self,value):
        if value < -273.15:
            self._celsius=-273.15
        else:
            self._celsius = value

    def __str__(self):
        return f'Temperature{self._celsius}°C ; {self.kelvin}K ; {self.fahrenheit}°F'

    @property
    def kelvin(self):
        return self._celsius + 273.15

    @kelvin.setter
    def kelvin(self,value):
        self.celsius = value - 273.15

    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self,value):
        self.celsius = (value - 32) * 5/9 
#wariant A
    @property
    def is_boiling(self):
        return self._celsius >= 100
    @property
    def is_freezing(self):
        return self._celsius <= 0
    #wariant B
    @property
    def state(self):
        if self._celsius >= 100:
            return 'gas'
        elif self._celsius <= 0:
            return 'solid'
        else:
            return 'liquid'

temp = Temperature(25)
print(temp) 
print(f'Start: {temp.celsius}°C')
print(f'Stan skupienia: {temp.state}')    
print(f'Czy zamarza? {temp.is_freezing}')

temp = Temperature(100)
print(temp) 
print(f'Start: {temp.celsius}°C')
print(f'Stan skupienia: {temp.state}')    
print(f'Czy wrze? {temp.is_boiling}')


Temperature25°C ; 298.15K ; 77.0°F
Start: 25°C
Stan skupienia: liquid
Czy zamarza? False
Temperature100°C ; 373.15K ; 212.0°F
Start: 100°C
Stan skupienia: gas
Czy wrze? True


### kolejność
1. uzytkownik daje t.kelvin = 50 -> skoro chce przypisać wartość to idzie do kelvin.setter
2. setter dostaje wartość 50 i robi obliczenia
3. skoro jest w nim self.celsius = -223.15 BEZ PODŁOGI TO SZUKA SETTERA DLA CELSIUS
4. setter celsius sprawdza wartość
5. zapisuje wartość do celsius czyli tego głównego z podłogą self._celsius = -223.15


# ZAD 3

In [18]:
def lcg(seed, a, c, m, N):
    x=seed # I wartość
    for _ in range(N):
        yield x
        x=(a * x + c) % m
       

#testowanie
seed = 1
a = 5
c = 3
m = 16
N = 10  

generator_reczny = lcg(seed, a, c, m, N)

for _ in range(min(3,N)): #zeby bledu nie bylo
    print(next(generator_reczny))

print('Pobieranie liczb z iterowania:')
for liczba in lcg(seed, a, c, m, N):
    print(liczba)

1
8
11
Pobieranie liczb z iterowania:
1
8
11
10
5
12
15
14
9
0


modulo 16 zwraca liczby od 0-15 bo reszte z dzielenia przez 15 daje
jak podzielimy wynik tego modulo przez 16 czyli m to zawsze dzielimy cos mniejszego od 15 przez 16. 

In [19]:
def lcg_float(seed, a, c, m, N): #[0,1)
    x = seed 
    for _ in range(N):
        yield x / m       
        x = (a * x + c) % m
        

for liczba in lcg_float(1, 5, 3, 16, 5):
    print(liczba)

0.0625
0.5
0.6875
0.625
0.3125
